# Nim Preliminary Sweep Analysis

DeepSeek-R1-Distill-7B playing Nim (3, 5, 7) with varying CoT budgets.

Two sweep configs:
- `nim_prelim` — LLM vs **Random** opponent (floor: can LLM beat random?)
- `nim_vs_optimal` — LLM vs **NimOptimal** (gap from perfect play)

Oracle: `NimOptimalAgent` — exact XOR policy, win_rate ∈ {0, 0.5, 1.0}.

Key metrics:
- **Win rate** vs opponent
- **Oracle optimal rate** — fraction of turns where model picks a nim-winning move
- **Move regret** — 0 for optimal moves, 0.5 for suboptimal moves (from nim oracle)
- **Move regret by phase** (early/mid/late, cutoffs 4/12)


In [ ]:
import json
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..') if Path('..').joinpath('src').exists() else Path('.')
DB_PATH = ROOT / 'data' / 'results.db'

if not DB_PATH.exists():
    print(f'WARNING: {DB_PATH} not found.')
else:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    runs = pd.read_sql_query(
        "SELECT run_id, name, created_at FROM runs WHERE name LIKE 'nim%' ORDER BY created_at",
        con
    )
    print(f'Nim runs found: {len(runs)}')
    print(runs[['run_id','name','created_at']].to_string(index=False))


## Win Rate vs CoT Budget

In [ ]:
if 'con' in dir() and len(runs) > 0:
    trials = pd.read_sql_query("""
        SELECT t.trial_id, t.run_id, t.seed,
               r.name as run_name,
               json_extract(t.condition_json, '$.budget') as budget,
               json_extract(t.condition_json, '$.opponent') as opponent,
               t.winner
        FROM trials t
        JOIN runs r ON t.run_id = r.run_id
        WHERE r.name LIKE 'nim%'
    """, con)
    if len(trials) == 0:
        print('No trial data yet.')
    else:
        trials['budget'] = trials['budget'].astype(int)
        trials['llm_win'] = (trials['winner'] == 'llm').astype(float)
        fig, axes = plt.subplots(1, max(1, trials['run_name'].nunique()), figsize=(14, 5), sharey=True)
        if trials['run_name'].nunique() == 1:
            axes = [axes]
        for ax, (rname, grp) in zip(axes, trials.groupby('run_name')):
            wr = grp.groupby('budget')['llm_win'].agg(['mean','count']).reset_index()
            wr.columns = ['budget','win_rate','n']
            ax.plot(wr['budget'], wr['win_rate']*100, 'o-', linewidth=2, markersize=8)
            ax.axhline(50, color='gray', linestyle=':', alpha=0.5, label='50% baseline')
            ax.set_xlabel('CoT Budget (tokens)')
            ax.set_ylabel('LLM Win Rate (%)')
            ax.set_title(rname)
            ax.set_ylim(-5, 105)
            ax.legend()
            for _, row in wr.iterrows():
                ax.annotate(f'{row["win_rate"]*100:.0f}% (N={row["n"]})',
                            (row['budget'], row['win_rate']*100+2), ha='center', va='bottom', fontsize=9)
        plt.suptitle('Nim: LLM Win Rate vs CoT Budget', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(ROOT/'data'/'nim_win_rate.png', dpi=150, bbox_inches='tight')
        plt.show()


## Oracle Optimal Rate and Move Regret

In [ ]:
if 'con' in dir() and len(runs) > 0:
    turns = pd.read_sql_query("""
        SELECT t.turn_id, t.trial_id, t.turn_idx, t.phase, t.agent_kind,
               t.move_quality, t.move_regret, t.oracle_chosen_rank, t.n_legal_moves,
               tr.run_id,
               r.name as run_name,
               json_extract(tr.condition_json, '$.budget') as budget
        FROM turns t
        JOIN trials tr ON t.trial_id = tr.trial_id
        JOIN runs r ON tr.run_id = r.run_id
        WHERE r.name LIKE 'nim%'
          AND t.agent_kind = 'llm'
    """, con)
    if len(turns) == 0:
        print('No turn data yet.')
    else:
        turns['budget'] = turns['budget'].astype(int)
        # Regret: for nim oracle, 0.0 = optimal, 0.5 = suboptimal (losing position)
        agg = turns.groupby(['run_name','budget']).agg(
            n_turns=('turn_id','count'),
            opt_rate=('move_quality','mean'),
            avg_regret=('move_regret','mean'),
        ).reset_index()
        print('LLM turn metrics by budget:\n')
        print(agg.to_string(index=False))


In [ ]:
if 'con' in dir() and len(runs) > 0 and 'turns' in dir() and len(turns) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    markers = {'nim_prelim': 'o-', 'nim_vs_optimal': 's--'}
    colors = {'nim_prelim': 'steelblue', 'nim_vs_optimal': 'darkorange'}
    for rname, grp in turns.groupby('run_name'):
        agg = grp.groupby('budget').agg(
            opt_rate=('move_quality','mean'), avg_regret=('move_regret','mean')
        ).reset_index()
        mk = markers.get(rname, 'o-')
        col = colors.get(rname, 'gray')
        ax1.plot(agg['budget'], agg['opt_rate']*100, mk, color=col, linewidth=2, label=rname)
        ax2.plot(agg['budget'], agg['avg_regret'], mk, color=col, linewidth=2, label=rname)
    ax1.set_xlabel('Budget'); ax1.set_ylabel('Optimal play rate (%)')
    ax1.set_title('Oracle Optimal Rate (LLM turns)'); ax1.legend()
    ax2.set_xlabel('Budget'); ax2.set_ylabel('Avg move regret')
    ax2.set_title('Average Move Regret'); ax2.legend()
    plt.tight_layout()
    plt.savefig(ROOT/'data'/'nim_quality.png', dpi=150, bbox_inches='tight')
    plt.show()


## Regret by Game Phase

In [ ]:
if 'turns' in dir() and len(turns) > 0:
    phase_order = ['early', 'mid', 'late']
    phase_agg = turns.groupby(['budget','phase']).agg(
        avg_regret=('move_regret','mean'), n=('turn_id','count')
    ).reset_index()
    phase_agg = phase_agg[phase_agg['phase'].isin(phase_order)]
    fig, ax = plt.subplots(figsize=(10, 4))
    budgets = sorted(turns['budget'].unique())
    x = np.arange(len(budgets))
    width = 0.25
    colors = {'early':'lightblue', 'mid':'steelblue', 'late':'navy'}
    for i, phase in enumerate(phase_order):
        grp = phase_agg[phase_agg['phase']==phase].set_index('budget').reindex(budgets)
        ax.bar(x + i*width - width, grp['avg_regret'].fillna(0), width, label=phase, color=colors[phase])
    ax.set_xticks(x)
    ax.set_xticklabels([f'B={b}' for b in budgets])
    ax.set_ylabel('Average move regret')
    ax.set_title('Move Regret by Game Phase')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT/'data'/'nim_regret_phase.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if 'con' in dir():
    con.close()
    print('Connection closed.')
